In [1]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)
print("Added to sys.path:", assignment_root)

from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/mayurakshi/Documents/Model To Markets/FRE-GY-9743-Assignments-1
Fixed Income Library is loaded.


## Homework 1 --- 1-D Interpolation

Implement the four methods marked `## TODO` inside `Interpolator1DPCP`, in
`fixedincomelib/utilities/numerics.py`:

- `interpolate`
- `integrate`
- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

Then fill in `bump_reval_interpolator_integrand` further down in this notebook.

The interpolation convention is spelled out in the `Interpolator1DPCP`
docstring. Read it before writing.

To check yourself, run every cell in this notebook top to bottom. Each check
prints your value next to the expected one --- every `diff` should be around `0.0`.


### Test interpolation

In [6]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3, expected 3.0, diff = 0.0
f(1.0) = 3, expected 3.0, diff = 0.0
f(1.5) = 4, expected 4.0, diff = 0.0
f(3.0) = 4, expected 4.0, diff = 0.0
f(5.5) = 6, expected 6.0, diff = 0.0
f(6.5) = 6, expected 6.0, diff = 0.0
f(8.0) = 6, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [7]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [9]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1DIntegral(x_s, x_e, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)

### Interpolation sensitivity

In [10]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [11]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 7.571721027943568e-12
[1.5, 5.2]: max abs diff = 2.7955415760061442e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 1.1823431123048067e-10


# Answers: Open Questions

# Part A: Bond Forward Business

**Setup.** Bank A sells a 1-year forward on a US Treasury maturing at $T_m$ (10 years from inception) to a client, notional 1,000,000. The payoff to the buyer at settlement is

$$V(T_s) = B(T_s;T_s,T_m) - K,$$

and the strike $K$ is set so that the contract has zero value at inception, i.e. $K = B(0;T_s,T_m)$.

---

## Q1. Client motivation, term sheet, and settlement

### Why would the client enter the trade?

Treasuries have negligible default risk, but their **market price is exposed to interest-rate risk**: a 10-year Treasury has a modified duration of roughly 8, so a 1% rise in yields lowers its price by about 8%. The client is the *buyer* (long) of the forward, and the likely motivations are:

1. **Hedging a future purchase.** A pension fund or insurer expecting cash in one year can lock in today's purchase price $K$ and remove the risk that yields fall (prices rise) before it invests.
2. **No upfront cash.** Buying the bond today would require funding today; a forward requires no cash outlay until $T_s$.
3. **Speculation / leverage.** The client can take a view that rates will fall (prices will rise) without paying for the bond upfront.

### Key terms on the term sheet

| Term | Description |
|---|---|
| Parties | Client (buyer, long forward); Bank A (seller, short forward) |
| Trade date / settlement date | Inception $t=0$; settlement $T_s$ (1 year later) |
| Notional | 1,000,000 face value |
| Underlying | The specific US Treasury (CUSIP, coupon, maturity $T_m$) |
| Strike $K$ | Agreed forward price (per 100 face), set so that $V(0)=0$ |
| Price convention | Clean or dirty; treatment of accrued interest |
| Settlement type | Physical delivery of the bond, or cash settlement of $B(T_s)-K$ |
| Conventions | Day count, business-day rules, coupon dates |
| CSA / collateral | Determines the discount curve $df_{csa}$ used in the MTM |

The underlying must be identified exactly because different Treasuries have very different prices. Note that $T_m$ is a fixed maturity date: at $T_s$ the delivered bond has $T_m - T_s \approx 9$ years remaining.

### What happens on the settlement date $T_s$ (physical settlement)

- **Bank A delivers** the bond (1,000,000 face) to the client.
- **The client pays** $K$ per 100 face, i.e. $\text{Notional}\times K/100$, plus accrued interest if $K$ is quoted clean.

The client therefore acquires the bond at exactly the price fixed at inception, regardless of where the market trades on $T_s$.

Under cash settlement, only the net amount $\text{Notional}\times\big(B(T_s)-K\big)/100$ is exchanged: Bank A pays the client if it is positive, and the client pays Bank A if it is negative.

---

## Q2. Hedging with the spot bond, and internal funding

### Why buy the spot bond at $t=0$?

Bank A is **short** the forward: on $T_s$ it must deliver the bond and receive $K$, so its payoff is

$$K - B(T_s;T_s,T_m).$$

This loses money if bond prices rise. The desk removes this exposure by **buying the bond today** at $B(0;0,T_m)$ and holding it until $T_s$:

- If the bond price rises, the loss on the forward is offset by the gain on the bond held.
- If the bond price falls, the gain on the forward is offset by the loss on the bond held.

Since the bank already owns the bond, it can deliver it on $T_s$ regardless of where the market trades. This is a **static hedge** (buy once, hold, no rebalancing) and requires no model of interest-rate dynamics. It is also the reason the forward can be priced by cost of replication: the fair strike is what it costs the bank to buy, finance, and hold the bond until $T_s$.

### Funding the bond position (internal trades)

Buying the bond requires cash of $B(0;0,T_m)$ at $t=0$, which the desk does not want to fund from its own capital. The funding is arranged internally:

1. **Treasury desk** is the bank's source of cash.
2. **Repo desk** lends cash against collateral. The trading desk enters a term repo with it: it delivers the bond as collateral, receives cash equal to the bond's price (haircut ignored), and agrees to repurchase it on $T_s$ for principal plus interest at the term repo rate $R$.
3. The cash received is used to pay for the bond purchase, so the **net cash outlay at $t=0$ is zero**.

### Cash flows of the full structure

| Time | Bond position | Repo / funding | Forward with client |
|---|---|---|---|
| $t=0$ | Buy bond, pay $B(0;0,T_m)$ | Receive $B(0;0,T_m)$ against the bond as collateral | No cash exchanged |
| $t_i \le T_s$ | Receive coupon $c_i$ | Coupon passes to the repo desk and reduces the debt | None |
| $T_s$ | Bond returned by the repo desk | Repay principal plus interest at rate $R$, net of coupons | Deliver bond, receive $K$ |

At $T_s$ the desk's total profit is $K - B(0;T_s,T_m)$, where $B(0;T_s,T_m)$ is the repo repayment net of coupons (derived in Q3). This does not depend on the market price $B(T_s)$, so the desk holds no bond-price risk and no financing-rate risk (the repo rate is fixed at inception for the full term).

---

## Q3. Forward price from the repo rate

**Inputs:** the coupon schedule $\{(t_i, c_i)\}$ of the Treasury, the term repo rate $R$ for $[0,T_s]$, and the spot price $B(0;0,T_m)$. All prices are per 100 face and are **dirty** (clean price plus accrued interest), since the dirty price is the cash actually paid for the bond and financed in the repo.

### Step 1: Repayment of the repo loan

Borrowing \$1 at the repo rate $R$ for the period $[0,T_s]$ requires repaying

$$1 + R\,T_s$$

at $T_s$ (simple interest, with $T_s$ as the accrual fraction, e.g. ACT/360). Borrowing $B(0;0,T_m)$ therefore requires repaying $B(0;0,T_m)\,(1+R\,T_s)$ at $T_s$ if there were no coupons.

### Step 2: Effect of the coupons

The bond is held as collateral by the repo desk, so any coupon $c_i$ paid at $t_i \le T_s$ goes to the repo desk. **It reduces the debt**, and because it is received early, it also stops accruing interest from $t_i$ to $T_s$. Its value in $T_s$ terms is

$$c_i\,\big(1 + R\,(T_s - t_i)\big).$$

### Step 3: Forward price

The bank's net cost of buying, financing, and holding the bond until $T_s$ is the loan repayment minus the coupons:

$$\boxed{B(0;T_s,T_m) = B(0;0,T_m)\,(1+R\,T_s) \;-\; \sum_{t_i \le T_s} c_i\,\big(1+R\,(T_s-t_i)\big)}$$

The strike is $K = B(0;T_s,T_m)$, so that $V(0) = 0$.

**Why this is the right price (no-arbitrage argument).** The bank can replicate the forward exactly (buy the bond, fund with repo, hold until $T_s$) at this net cost. If $K$ were higher, the bank would earn a riskless profit by selling the forward and running the replication. If $K$ were lower, the client would be able to lock in a riskless profit by buying the forward and doing the reverse.

### Numerical example

Take $B(0;0,T_m)=100$, $R=4\%$, $T_s=1$, and a single coupon $c=2$ paid at $t_1=0.5$:

- Loan repayment: $100 \times (1 + 0.04\times 1) = 104.00$
- Coupon value at $T_s$: $2 \times (1 + 0.04 \times 0.5) = 2.04$
- Forward price: $104.00 - 2.04 = \mathbf{101.96}$

### Sensitivities (sanity checks)

- Higher repo rate $R$ raises the forward price: the bank pays more interest on its funding and passes that cost to the client.
- Larger or earlier coupons lower the forward price: the bank collects more cash while holding the bond, which pays down the loan.

### Remarks

- **Clean quote.** If $K$ is quoted clean, subtract the accrued interest at $T_s$: $K_{\text{clean}} = B(0;T_s,T_m) - AI(T_s)$.
- **Compounding.** Term repo is normally simple interest. If the repo rate is instead quoted with continuous compounding, replace $1+R\,\tau$ by $e^{R\,\tau}$ throughout.

---

## Q4. Mark-to-market during the life of the trade, $t \in (0, T_s]$

The strike $K$ is fixed in the contract. What changes over time is the **fair forward price** for the same delivery on $T_s$, because the spot bond price, the repo rate, and the set of remaining coupons all change. The MTM is the difference between the current fair forward price and $K$, discounted from $T_s$ back to $t$.

### Step 1: Recompute the forward price at time $t$

Apply the Q3 replication argument from $t$ instead of $0$ (buy the bond at today's market price, fund it with repo until $T_s$, collect the remaining coupons):

$$B(t;T_s,T_m) = B(t;t,T_m)\,\big(1+R_t\,(T_s-t)\big) \;-\; \sum_{t < t_i \le T_s} c_i\,\big(1+R_t\,(T_s-t_i)\big)$$

- $B(t;t,T_m)$ is the current market (dirty) price of the bond.
- $R_t$ is the current repo rate for the remaining period $[t,T_s]$.
- Only coupons still to be paid ($t < t_i \le T_s$) enter the sum. Coupons paid before $t$ are already in the past and are no longer part of the deal.

### Step 2: Compute the value

$$\boxed{V(t) = df_{csa}(t,T_s)\,\big(B(t;T_s,T_m) - K\big)}$$

The payoff $B(t;T_s,T_m)-K$ is only exchanged on $T_s$, so it is discounted to $t$ using the CSA discount factor $df_{csa}(t,T_s)$. This is equation (2) of the assignment.

### Interpretation

- $V(t) > 0$: the fair forward price has risen above $K$, so the client (who buys at $K$) is in the money and Bank A owes value.
- $V(t) < 0$: the fair forward price has fallen below $K$, so the client is out of the money and Bank A is in the money.
- The bank's forward position is the mirror image, $-V(t)$. This is offset by the gain or loss on the bond and repo positions it holds as a hedge, so the desk's total P&L stays close to zero.
- Consistency check: at $t=0$, $K = B(0;T_s,T_m)$ gives $V(0)=0$.

### Numerical example

Continue the Q3 example ($K=101.96$, $R=4\%$, one coupon of 2 at $t=0.5$, $T_s=1$). At $t=0.6$ the coupon has already been paid, so the sum is empty. Suppose the bond now trades at $B(t;t,T_m)=103$, the repo rate is still $4\%$, and the remaining term is $T_s-t=0.4$. Take $df_{csa}(0.6,1)\approx 1/(1+0.04\times0.4)\approx 0.984$.

- Forward price: $103\times(1+0.04\times 0.4)=104.65$
- Gap to strike: $104.65-101.96=2.69$
- Value: $V(0.6)\approx 0.984\times 2.69\approx \mathbf{+2.65}$ to the client, and $-2.65$ to Bank A

If instead the bond had fallen to $97$, the forward price would be $98.55$, the gap $-3.41$, and $V(0.6)\approx -3.35$ to the client (a gain to Bank A).

---

## Q5. Market risk, and how the desk earns revenue

### Is the desk exposed to major market risk?

**No.** If the desk follows Q2 and Q3, every source of market risk is neutralised at inception:

| Risk | How it is removed |
|---|---|
| Bond price / interest-rate risk | The desk owns the bond, so the loss on the short forward is offset by the gain on the bond, and vice versa |
| Financing-rate risk | The term repo fixes the borrowing rate $R$ for the full period $[0,T_s]$ |
| Strike risk | $K$ is fixed in the contract |

The desk's payoff at $T_s$ is $K - B(0;T_s,T_m)$, which is known at $t=0$ and does not depend on $B(T_s)$.

The desk still carries **small residual risks**: counterparty (credit) risk on the client, haircut and margin requirements on the repo, basis between the repo curve and the CSA discount curve, and operational or settlement risk.

### Does this mean the desk makes no money?

At the fair strike $K = B(0;T_s,T_m)$, **yes, the expected profit is zero**. The desk buys the bond, funds it, holds it, and delivers it at exactly its cost. A riskless, zero-cost replication earns nothing, so the desk needs a **spread**. This is the same reason a market maker quotes a bid-offer spread around fair value.

### How to earn revenue: quote with a marked-up repo rate

$K$ increases with the repo rate $R$, because higher financing costs are passed on to the client. The desk therefore quotes the client using a repo rate above its actual funding rate:

$$R_{\text{quote}} = R + s, \qquad s > 0,$$

$$K_{\text{quote}} = B(0;0,T_m)\,(1+R_{\text{quote}}\,T_s) - \sum_{t_i \le T_s} c_i\,\big(1+R_{\text{quote}}\,(T_s-t_i)\big)$$

The desk still funds itself at the true rate $R$. The client pays $K_{\text{quote}} > K_{\text{fair}}$ and the difference is the desk's locked-in revenue:

$$\text{Revenue at } T_s = \frac{\text{Notional}}{100}\,\big(K_{\text{quote}} - K_{\text{fair}}\big).$$

**Numerical example** (Q3 inputs: $B(0;0,T_m)=100$, one coupon of 2 at $t=0.5$, $T_s=1$):

- Fair strike at $R=4.0\%$: $K_{\text{fair}} = 100(1.04) - 2(1.02) = 101.96$
- Quoted strike at $R_{\text{quote}}=4.3\%$: $K_{\text{quote}} = 100(1.043) - 2(1.0215) = 102.257$
- Spread earned: $0.297$ per 100 face, so $\frac{1{,}000{,}000}{100}\times 0.297 \approx \$2{,}970$ on the 1,000,000 notional

The revenue is known at inception and carries no market risk, because the hedge is unchanged. Only $K$ differs.

### How to justify the spread to the client

1. **Price certainty with no upfront cash.** The client fixes its purchase price for the year without paying for the bond today.
2. **The bank provides a service.** It buys the bond, arranges the financing, and runs the hedge. It also uses its balance sheet and carries counterparty and operational risk. The spread is the fee for these services.
3. **The cost is small compared with the risk removed.** A spread of about 0.3 per 100 face is tiny next to the roughly 8% price move (about 8 per 100) a 1% change in yields would cause on a 10-year Treasury.
4. **Market convention.** Dealers always quote forwards with a bid-offer, which here appears as a markup on the financing rate.

---

## \*Q5. Link between the replication argument (Q2 to Q3) and risk-neutral pricing

**Claim.** The desk's "buy the bond, fund with repo" price is exactly the risk-neutral (change-of-numeraire) forward price, provided the repo curve is the discount curve.

### Notation

- $P(t,T)$: discount factor (zero-coupon bond price) from the funding curve.
- $B(t) := B(t;t,T_m)$: dirty price of the bond at $t$, *after* any coupons paid at or before $t$.
- $c_i$: deterministic coupons (fixed-rate Treasury) paid at $t_i$.

### Step 1: Find a traded asset that has no coupons before $T_s$

Define

$$Y(t) = B(t) - \sum_{t < t_i \le T_s} c_i\,P(t,t_i).$$

$Y$ is the value of a portfolio that is long the bond and short zero-coupon bonds maturing on the coupon dates, so it is a **traded asset**. Its coupons cancel against the short zeros. At maturity $Y(T_s) = B(T_s)$, since no coupons remain after $T_s$.

### Step 2: Use the forward-measure martingale property

Take $P(t,T_s)$ as the numeraire, with associated measure $\mathbb{Q}^{T_s}$. A traded asset divided by the numeraire is a martingale, so

$$\frac{Y(0)}{P(0,T_s)} = \mathbb{E}^{T_s}\!\left[\frac{Y(T_s)}{P(T_s,T_s)}\right] = \mathbb{E}^{T_s}\big[B(T_s)\big].$$

### Step 3: Price the forward

The value of the forward at $t=0$ is $V(0) = P(0,T_s)\,\mathbb{E}^{T_s}\big[B(T_s) - K\big]$. Setting $V(0)=0$ gives $K = \mathbb{E}^{T_s}[B(T_s)]$, and by Step 2:

$$\boxed{K = B(0;T_s,T_m) = \frac{B(0;0,T_m) - \sum_{t_i \le T_s} c_i\,P(0,t_i)}{P(0,T_s)}}$$

### Step 4: Match with the repo formula from Q3

If the repo rate is the discount rate, then

$$\frac{1}{P(0,T_s)} = 1 + R\,T_s, \qquad \frac{P(0,t_i)}{P(0,T_s)} = 1 + F(t_i,T_s)\,(T_s - t_i),$$

where $F(t_i,T_s)$ is the forward repo rate implied by the curve. Substituting:

$$B(0;T_s,T_m) = B(0;0,T_m)\,(1 + R\,T_s) - \sum_{t_i \le T_s} c_i\,\big(1 + F(t_i,T_s)\,(T_s - t_i)\big),$$

which is the Q3 formula. Under a flat rate $R$ with continuous compounding the match is exact: $1/P(0,T_s) = e^{R T_s}$ and $P(0,t_i)/P(0,T_s) = e^{R(T_s - t_i)}$.

| Risk-neutral formula | Replication (Q3) |
|---|---|
| $B(0;0,T_m)/P(0,T_s)$ | Spot price grown at the repo rate: $B(0;0,T_m)(1+R\,T_s)$ |
| $c_i\,P(0,t_i)/P(0,T_s)$ | Coupon credited to the repo desk and grown to $T_s$: $c_i(1+R(T_s-t_i))$ |
| Short zero-coupon bonds in $Y(t)$ | Borrowing through the term repo |

### Numerical check (continuous compounding)

With $B(0;0,T_m)=100$, $R=4\%$, $T_s=1$, and one coupon of 2 at $t=0.5$:

- Replication: $100\,e^{0.04} - 2\,e^{0.02} = 104.0811 - 2.0404 = 102.0407$
- Risk-neutral: $\dfrac{100 - 2\,e^{-0.02}}{e^{-0.04}} = \dfrac{98.0396}{0.960789} = 102.0407$

The two agree. This differs from the 101.96 in Q3 only because Q3 uses simple interest and this check uses continuous compounding.

### Same result for the MTM

For $t>0$, $F(t) = \mathbb{E}^{T_s}[B(T_s)\mid\mathcal{F}_t] = \big(B(t) - \sum_{t<t_i\le T_s} c_i P(t,t_i)\big)/P(t,T_s)$ and $V(t) = P(t,T_s)\,(F(t) - K)$. With $df_{csa}=P$, this is the Q4 formula.

### Assumptions needed for the two approaches to coincide

1. **Deterministic coupons and no default risk** on the bond, as for a Treasury.
2. **The repo curve equals the discount curve** (the CSA curve). There is no repo spread, no specialness, and no basis.
3. **Coupons are credited at the forward repo rate** implied by the same curve.
4. **No haircut, margin, or transaction costs**, and no arbitrage.

Interest rates are **not** required to be deterministic. Only the no-arbitrage argument and the change of numeraire are used.

### When they differ

If the repo rate differs from the discount rate, or the bond is special in repo, then $K$ from replication (using the actual repo rate) differs from the risk-neutral price by the **repo basis**. This is also the room the desk uses for its spread in Q5.